In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 🎯 Recruiting Pipeline Crew

## What We're Building

A 4-agent recruiting pipeline that screens candidates:

| Agent | Role |
|-------|------|
| **Resume Screener** | Parse resumes and score them against job requirements |
| **Interview Question Generator** | Create tailored technical + behavioral questions |
| **Candidate Summarizer** | Write a concise hiring manager brief |
| **Bias Checker** | Audit the pipeline for unconscious biases |

## Key CrewAI Concept: `async_execution`

The **Resume Screener** and **Interview Question Generator** can work in parallel
since they both read the same inputs independently. We use `async_execution=True`
on these tasks so they run concurrently, then the summarizer waits for both.

## Stack
- **CrewAI** — multi-agent pipeline
- **Ollama** — local LLM

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model="ollama/qwen3.5:9b", base_url="http://localhost:11434")
print(f"LLM ready: {llm.model}")

## Step 2 — Job Description & Candidate Data

In [ ]:
job_description = """
POSITION: Senior Backend Engineer
COMPANY: CloudScale Inc. (Series B, 120 employees)
LOCATION: Remote (US time zones)
SALARY: $160K-$200K + equity

REQUIREMENTS:
- 5+ years backend development (Python or Go preferred)
- Experience with distributed systems (microservices, message queues)
- Strong SQL + at least one NoSQL database
- Cloud infrastructure (AWS or GCP)
- CI/CD pipelines and containerization (Docker, Kubernetes)
- API design (REST or gRPC)

NICE TO HAVE:
- Experience with event-driven architecture (Kafka, RabbitMQ)
- Performance optimization at scale (>10K rps)
- Open-source contributions
- Tech lead or mentorship experience

CULTURE:
- Small, senior team — high autonomy, low meetings
- Ship fast, iterate, own your domain
- Code review culture, strong testing expectations
"""

candidate_resume = """
NAME: Priya Sharma
EMAIL: priya.sharma@email.com
LOCATION: Austin, TX

SUMMARY:
Backend engineer with 6 years of experience building scalable data pipelines
and microservices. Currently at a fintech startup processing 50K+ transactions/sec.

EXPERIENCE:

Senior Backend Engineer — PayFlow (fintech startup, 45 employees)
Jan 2022 — Present (3 years)
- Designed event-driven payment processing pipeline using Kafka + Python
- Reduced API latency from 200ms to 45ms by optimizing PostgreSQL queries
- Built real-time fraud detection service handling 50K transactions/sec
- Migrated monolith to 12 microservices on Kubernetes (AWS EKS)
- Mentored 3 junior engineers; led weekly architecture review sessions
- On-call rotation lead; reduced P1 incidents by 60%

Backend Engineer — DataVault (data analytics, 200 employees)
Jun 2019 — Dec 2021 (2.5 years)
- Built ETL pipelines processing 2TB/day using Python + Apache Airflow
- Designed REST APIs serving 15 internal teams (FastAPI + PostgreSQL)
- Implemented Redis caching layer, reducing DB load by 40%
- Wrote comprehensive test suites (pytest), maintained 92% code coverage

Junior Developer — TechStart Agency
Aug 2018 — May 2019 (10 months)
- Built client websites and APIs using Django + PostgreSQL
- First exposure to Docker and CI/CD (GitHub Actions)

EDUCATION:
B.S. Computer Science — UT Austin (2018)

SKILLS:
Python, Go (learning), PostgreSQL, Redis, MongoDB, Kafka, Docker, Kubernetes,
AWS (EKS, RDS, S3, Lambda), FastAPI, Django, gRPC, pytest, GitHub Actions,
Terraform, Datadog

OPEN SOURCE:
- Contributor to Apache Airflow (3 merged PRs — DAG serialization fixes)
- Maintainer of `py-kafka-tools` (450 GitHub stars)
"""

print("Job description and candidate resume loaded")

## Step 3 — Create Agents

In [ ]:
resume_screener = Agent(
    role="Resume Screener",
    goal="Score the candidate against job requirements with a detailed rubric",
    backstory=(
        "You screen 200+ resumes per week for senior engineering roles. You use "
        "a structured rubric: must-have requirements get pass/fail, nice-to-haves "
        "get bonus points. You look for evidence of impact (numbers, scale), not "
        "just keyword matches. You flag red flags AND green flags."
    ),
    llm=llm,
    verbose=True,
)

question_generator = Agent(
    role="Interview Question Generator",
    goal="Create tailored interview questions based on the candidate's background",
    backstory=(
        "You design structured interviews for senior engineering hires. For each "
        "candidate, you create questions that probe their claimed experience — "
        "if they say they optimized query latency, you ask HOW. You mix technical "
        "deep-dives, system design scenarios, and behavioral questions."
    ),
    llm=llm,
    verbose=True,
)

candidate_summarizer = Agent(
    role="Candidate Summarizer",
    goal="Write a concise hiring manager brief with a clear recommendation",
    backstory=(
        "You write the candidate summary that the hiring manager reads before "
        "deciding whether to schedule an interview. It must be scannable in 30 "
        "seconds: fit score, strengths, concerns, and a YES/NO/MAYBE recommendation "
        "with reasoning."
    ),
    llm=llm,
    verbose=True,
)

bias_checker = Agent(
    role="Hiring Bias Auditor",
    goal="Review the screening process for unconscious bias and ensure fairness",
    backstory=(
        "You're an I/O psychologist specializing in fair hiring practices. You "
        "audit screening decisions for common biases: affinity bias (similar "
        "background to interviewer), halo effect (one impressive thing overshadows "
        "gaps), pedigree bias (overvaluing school/company brands), and recency bias. "
        "You flag potential biases and suggest corrections."
    ),
    llm=llm,
    verbose=True,
)

print("4 agents created")

## Step 4 — Create Tasks

**Parallelism**: `screen_task` and `question_task` both have `async_execution=True`.
They run concurrently since they both read the same inputs independently.
The `summary_task` uses `context=[screen_task, question_task]` to wait for both.

In [ ]:
screen_task = Task(
    description=f"""Screen this candidate against the job requirements.

JOB DESCRIPTION:
{job_description}

CANDIDATE RESUME:
{candidate_resume}

Use this scoring rubric:

MUST-HAVE REQUIREMENTS (pass/fail each):
□ 5+ years backend dev
□ Python or Go
□ Distributed systems experience
□ SQL + NoSQL
□ Cloud infrastructure
□ CI/CD + containers
□ API design

NICE-TO-HAVE (0-2 points each):
□ Event-driven architecture
□ Performance at scale
□ Open-source contributions
□ Tech lead / mentorship

Also note:
- GREEN FLAGS: Evidence of impact, ownership, growth
- RED FLAGS: Gaps, inconsistencies, concerns
- OVERALL FIT SCORE: 0-100""",
    expected_output="Structured rubric evaluation with fit score.",
    agent=resume_screener,
    async_execution=True,
)

question_task = Task(
    description=f"""Generate tailored interview questions for this candidate.

JOB DESCRIPTION:
{job_description}

CANDIDATE RESUME:
{candidate_resume}

Create exactly:

1. TECHNICAL DEEP-DIVE (3 questions)
   - Probe their specific claims (Kafka pipeline, latency optimization, etc.)
   - Include follow-up probes for each

2. SYSTEM DESIGN (1 scenario)
   - Design a system relevant to our company's domain
   - Include evaluation criteria

3. BEHAVIORAL (3 questions)
   - STAR format (Situation, Task, Action, Result)
   - Focus on: conflict resolution, mentorship, handling production incidents

4. RED FLAG PROBES (2 questions)
   - Why leaving current role? (fintech → our company)
   - Go experience is listed as "learning" — how far along?

For each question, explain what a GOOD vs. WEAK answer looks like.""",
    expected_output="Structured interview question set with evaluation criteria.",
    agent=question_generator,
    async_execution=True,
)

summary_task = Task(
    description="""Write the Hiring Manager Brief.

Using the screening rubric and interview questions from above, write:

## CANDIDATE BRIEF — [Name]

### Recommendation: [STRONG YES / YES / MAYBE / NO]

### Fit Score: [X/100]

### 30-Second Summary
(3 sentences: who they are, why they fit, any concerns)

### Strengths (top 3)
### Concerns (top 3)
### Interview Focus Areas
(What to probe in the interview)

### Compensation Notes
(Are they likely within our range? Any leverage?)

Keep it under 300 words.""",
    expected_output="Concise hiring manager brief with recommendation.",
    agent=candidate_summarizer,
    context=[screen_task, question_task],
    markdown=True,
)

bias_task = Task(
    description="""Audit the entire screening process for bias.

Review the resume screening, interview questions, and candidate summary for:

1. AFFINITY BIAS: Are we favoring the candidate because they're similar to us?
2. HALO EFFECT: Is one impressive achievement overshadowing gaps?
3. PEDIGREE BIAS: Are we over-weighting school/company brands?
4. GENDER/NAME BIAS: Would this evaluation change with a different name?
5. RECENCY BIAS: Are we over-emphasizing the most recent role?
6. CONFIRMATION BIAS: Are we looking for evidence to confirm an initial impression?

For each potential bias found:
- Describe the bias
- Quote the specific language that reveals it
- Suggest a correction

End with an overall FAIRNESS SCORE (1-10) and recommendations.""",
    expected_output="Bias audit with fairness score and corrections.",
    agent=bias_checker,
)

print("4 tasks created")
print("Note: screen_task and question_task will run in parallel (async_execution=True)")

## Step 5 — Run the Crew

In [ ]:
recruiting_crew = Crew(
    agents=[resume_screener, question_generator, candidate_summarizer, bias_checker],
    tasks=[screen_task, question_task, summary_task, bias_task],
    process=Process.sequential,
    verbose=True,
)

print("Recruiting pipeline crew assembled!")
print("Tasks 1 & 2 run in parallel, task 3 waits for both, then task 4 runs.")
result = recruiting_crew.kickoff()
print("\n✅ Recruiting pipeline complete!")

In [ ]:
print("📋 BIAS AUDIT REPORT (final output):")
print("=" * 60)
print(result.raw)

In [ ]:
# View each stage of the pipeline
labels = ["Resume Screening", "Interview Questions", "Hiring Brief", "Bias Audit"]
for label, task_out in zip(labels, result.tasks_output):
    print(f"\n{'=' * 60}")
    print(f"📌 {label}")
    print("=" * 60)
    print(task_out.raw[:800])
    if len(task_out.raw) > 800:
        print("... (truncated)")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **`async_execution=True`** | Resume screening & question gen run in parallel |
| **`context=[task_a, task_b]`** | Summary task waits for both parallel tasks |
| **Structured rubric** | Pass/fail must-haves + scored nice-to-haves |
| **Bias auditing** | Dedicated agent checks for 6 types of unconscious bias |

## Parallel Execution Flow

```
Input (JD + Resume)
    ├── [PARALLEL] Resume Screener → Fit Score
    ├── [PARALLEL] Question Generator → Interview Qs
    │         ↓ (both complete)
    └── Candidate Summarizer → Hiring Brief
              ↓
         Bias Checker → Fairness Audit
```

## 🔧 Extensions

- **Batch screening**: Use `kickoff_for_each()` to screen 50 candidates
- **ATS integration**: Pull resumes from Greenhouse or Lever API
- **Scorecard comparison**: Rank multiple candidates side by side
- **Diversity metrics**: Track pipeline diversity at each stage

---

## 📚 CrewAI Series Summary (Notebooks 41-50)

| # | Project | Key CrewAI Concept |
|---|---------|-------------------|
| 41 | Startup Idea Validation | Agent, Task, Crew, Process basics |
| 42 | Content Creation | Sequential context passing (auto-chain) |
| 43 | Lead Generation | `context=[]` explicit task dependencies |
| 44 | Job Hunt Assistant | `allow_delegation=True` |
| 45 | Academic Research | `markdown=True` formatted output |
| 46 | Real Estate Analysis | Task `callback` functions |
| 47 | Product Launch | Large crews (5 agents), exec distillation |
| 48 | Competitive Intelligence | Threat categorization, BLUF memos |
| 49 | Customer Success | `output_file` auto-save results |
| 50 | Recruiting Pipeline | `async_execution` parallel tasks |